In [1]:
from PIL import Image
import numpy as np
import os
import matplotlib.pyplot as plt

In [2]:
import numpy as np
from PIL import Image
import os

def gerar_mascara_oam(largura, altura, l, periodo_rede_x=0, periodo_rede_y=0):
    """
    Gera uma máscara de fase OAM com uma rede de difração opcional.
    
    Parâmetros:
    largura, altura : Resolução do seu SLM (ex: 1920, 1080)
    l               : Carga topológica (estado OAM)
    periodo_rede_x  : Período da rede no eixo X (em pixels). 0 para desativar.
    periodo_rede_y  : Período da rede no eixo Y (em pixels). 0 para desativar.
    """
    # Cria uma malha de coordenadas centralizada
    x = np.linspace(-largura // 2, largura // 2 - 1, largura)
    y = np.linspace(-altura // 2, altura // 2 - 1, altura)
    X, Y = np.meshgrid(x, y)
    
    # Converte para coordenadas polares (ângulo azimutal)
    theta = np.arctan2(Y, X)
    
    # 1. Fase Helicoidal (OAM)
    fase_total = l * theta
    
    # 2. Adiciona a rede de difração linear (Blazed Grating)
    # Isso move o feixe para fora do eixo principal para evitar a ordem zero do SLM
    if periodo_rede_x != 0:
        fase_total += (2 * np.pi / periodo_rede_x) * X
    if periodo_rede_y != 0:
        fase_total += (2 * np.pi / periodo_rede_y) * Y
        
    # 3. Faz o "Wrap" da fase para o intervalo [0, 2*pi]
    fase_wrapped = np.mod(fase_total, 2 * np.pi)
    
    # 4. Mapeia de [0, 2*pi] para valores inteiros de 8-bits [0, 255]
    # Nota: Verifique a calibração (Look-Up Table - LUT) do seu SLM da Thorlabs 
    # para garantir que 255 corresponde exatamente a 2*pi no comprimento de onda do seu laser.
    imagem_slm = (fase_wrapped / (2 * np.pi) * 255).astype(np.uint8)
    
    return imagem_slm

def pipeline_geracao_mascaras():
    # Configurações do SLM da Thorlabs (ajuste para o seu modelo, ex: Exulus)
    SLM_LARGURA = 1920
    SLM_ALTURA = 1080
    
    # Configurações da rede de difração (ajuste conforme a separação angular necessária)
    PERIODO_X = 50 
    
    # Pasta de saída
    pasta_saida = "mascaras_oam"
    os.makedirs(pasta_saida, exist_ok=True)
    
    # Gerar máscaras para estados OAM de l = -5 até l = 5
    estados_l = range(-5, 6)
    
    for l in estados_l:
        print(f"Gerando máscara para l = {l}...")
        mascara = gerar_mascara_oam(SLM_LARGURA, SLM_ALTURA, l, periodo_rede_x=PERIODO_X)
        
        # Salva como BMP ou PNG (BMP geralmente é recomendado para não ter compressão no SLM)
        nome_arquivo = os.path.join(pasta_saida, f"OAM_l{l}_grating.bmp")
        Image.fromarray(mascara).save(nome_arquivo)
        
    print("Pipeline concluído!")

if __name__ == "__main__":
    pipeline_geracao_mascaras()

Gerando máscara para l = -5...
Gerando máscara para l = -4...
Gerando máscara para l = -3...
Gerando máscara para l = -2...
Gerando máscara para l = -1...
Gerando máscara para l = 0...
Gerando máscara para l = 1...
Gerando máscara para l = 2...
Gerando máscara para l = 3...
Gerando máscara para l = 4...
Gerando máscara para l = 5...
Pipeline concluído!


In [1]:
import numpy as np
from PIL import Image
import os
import math

def gerar_mascara_lg_p0(largura, altura, l, w0, periodo_rede_x=50):
    """
    Gera máscara Laguerre-Gauss (p=0) combinando Fase e Amplitude num SLM de fase.
    """
    x = np.linspace(-largura // 2, largura // 2 - 1, largura)
    y = np.linspace(-altura // 2, altura // 2 - 1, altura)
    X, Y = np.meshgrid(x, y)
    
    R = np.sqrt(X**2 + Y**2)
    Theta = np.arctan2(Y, X)
    
    # 1. Equação de Amplitude Laguerre-Gauss (p=0)
    # A(r) = (r * sqrt(2) / w0)^|l| * exp(-r^2 / w0^2)
    termo_radial = (R * np.sqrt(2) / w0) ** abs(l)
    termo_gaussiano = np.exp(-(R**2) / (w0**2))
    amplitude = termo_radial * termo_gaussiano
    
    # Normaliza a amplitude entre 0 e 1
    if np.max(amplitude) > 0:
        amplitude = amplitude / np.max(amplitude)
        
    # 2. Fase OAM (Espiral)
    fase_oam = l * Theta
    
    # 3. Fase da Rede de Difração (Blazed Grating)
    fase_rede = (2 * np.pi / periodo_rede_x) * X
    
    # 4. Codificação de Amplitude e Fase (Método de Davis simplificado)
    # A amplitude controla a profundidade (contraste) da fase enrolada.
    # Onde a amplitude é 0, a fase é plana (luz vai para a ordem zero).
    # Onde a amplitude é 1, o garfo é completo (luz vai para a 1ª ordem OAM).
    fase_total_wrapped = np.mod(fase_oam + fase_rede, 2 * np.pi)
    
    # Modulação final: O "f(A)" linear.
    mascara_slm_fase = amplitude * fase_total_wrapped
    
    # Converte para 8-bits
    imagem_slm = (mascara_slm_fase / (2 * np.pi) * 255).astype(np.uint8)
    
    return imagem_slm

# Exemplo de uso
# w0 é o "waist" (cintura) do feixe gaussiano em pixels. Ajuste para caber na tela do seu SLM.
mascara_l3 = gerar_mascara_lg_p0(1920, 1080, l=3, w0=300, periodo_rede_x=40)
Image.fromarray(mascara_l3).save("LG_p0_l3_amplitude_modificada.bmp")